# 08 — OLID-I Full Feature Extraction Pipeline

**Project:** Graph-Based Representation of Plant Leaf Images for Fine-Grained Plant Stress Classification Using Weisfeiler-Lehman Kernels  
**Dataset:** OLID-I (Full Dataset: ~4,749 images)

**Purpose:**  
Scale the validated pilot pipeline to the entire dataset. Extract a robust set of phenotypic features encompassing **Shape**, **Global Color (RGB/HSV/Lab)**, **Texture (GLCM)**, and **Spatial/Regional Grids**. Ensure strict quality control and leakage-safe preprocessing.


In [1]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from skimage.feature import graycomatrix, graycoprops
import warnings

warnings.filterwarnings('ignore')

# Configuration
if os.path.exists('/content'):
    DATASET_PATH = '/content/OLID-I'
    OUTPUT_CSV   = '/content/olid_full_features.csv'
else:
    # Set to the real full dataset path.
    DATASET_PATH = r'C:/Users/vamsi/Documents/WL_Graph_Representation/dataset'
    OUTPUT_CSV   = 'olid_full_features.csv'

def extract_crop_condition(class_name: str):
    if '__' in class_name:
        crop, cond = class_name.split('__', 1)
        return crop, cond
    return class_name, 'unknown'


C:\Users\vamsi\AppData\Roaming\Python\Python312\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\vamsi\AppData\Roaming\Python\Python312\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
def segment_leaf(img_bgr):
    """
    Performs ExG + Otsu segmentation.
    Returns: (segmented_rgb, mask, mask_area_ratio, quality_flag)
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    R = img_rgb[:,:,0].astype(np.float32)
    G = img_rgb[:,:,1].astype(np.float32)
    B = img_rgb[:,:,2].astype(np.float32)
    
    denom = R + G + B
    denom[denom == 0] = 1
    
    exg = 2*(G/denom) - (R/denom) - (B/denom)
    exg_norm = cv2.normalize(exg, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    _, mask = cv2.threshold(exg_norm, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    final_mask = np.zeros_like(mask)
    if contours:
        main_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(final_mask, [main_contour], -1, 255, cv2.FILLED)
        
    mask_ratio = np.sum(final_mask > 0) / final_mask.size
    
    if mask_ratio == 0:
        quality_flag = 'FAILED'
    elif mask_ratio > 0.98:
        quality_flag = 'SUSPICIOUS'
    elif mask_ratio < 0.01:
        quality_flag = 'FAILED'
    else:
        quality_flag = 'ACCEPT'
        
    segmented_rgb = cv2.bitwise_and(img_rgb, img_rgb, mask=final_mask)
    return segmented_rgb, final_mask, mask_ratio, quality_flag


In [3]:
def extract_all_features(img_rgb, mask):
    """
    Extracts shape, color, texture, and spatial features from a segmented leaf.
    Returns a dictionary of features.
    """
    features = {}
    
    # 1. Shape & Geometry
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, True)
    x, y, w, h = cv2.boundingRect(cnt)
    
    features['area'] = area
    features['perimeter'] = perimeter
    features['aspect_ratio'] = float(w) / h if h > 0 else 0
    features['circularity'] = (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0
    features['extent'] = float(area) / (w * h) if w * h > 0 else 0
    
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    features['solidity'] = float(area) / hull_area if hull_area > 0 else 0
    features['convex_area'] = hull_area
    
    if len(cnt) >= 5:
        (cx, cy), (MA, ma), angle = cv2.fitEllipse(cnt)
        features['major_axis_length'] = max(MA, ma)
        features['minor_axis_length'] = min(MA, ma)
        a, b = max(MA, ma)/2, min(MA, ma)/2
        features['eccentricity'] = np.sqrt(1 - (b**2)/(a**2)) if a > 0 else 0
    else:
        features['major_axis_length'] = 0
        features['minor_axis_length'] = 0
        features['eccentricity'] = 0

    # 2. Color (Global)
    pixels = img_rgb[mask > 0]
    if len(pixels) == 0:
        return None
        
    R, G, B = pixels[:, 0], pixels[:, 1], pixels[:, 2]
    features['mean_R'], features['std_R'] = np.mean(R), np.std(R)
    features['mean_G'], features['std_G'] = np.mean(G), np.std(G)
    features['mean_B'], features['std_B'] = np.mean(B), np.std(B)
    
    img_hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    hsv_pixels = img_hsv[mask > 0]
    features['mean_H'], features['std_H'] = np.mean(hsv_pixels[:, 0]), np.std(hsv_pixels[:, 0])
    features['mean_S'], features['std_S'] = np.mean(hsv_pixels[:, 1]), np.std(hsv_pixels[:, 1])
    features['mean_V'], features['std_V'] = np.mean(hsv_pixels[:, 2]), np.std(hsv_pixels[:, 2])

    img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
    lab_pixels = img_lab[mask > 0]
    features['mean_L'], features['std_L'] = np.mean(lab_pixels[:, 0]), np.std(lab_pixels[:, 0])
    features['mean_a'], features['std_a'] = np.mean(lab_pixels[:, 1]), np.std(lab_pixels[:, 1])
    features['mean_b'], features['std_b'] = np.mean(lab_pixels[:, 2]), np.std(lab_pixels[:, 2])

    denom = (R.astype(float) + G.astype(float) + B.astype(float))
    denom[denom == 0] = 1
    r, g, b = R / denom, G / denom, B / denom
    features['green_ratio'] = np.mean(g)
    features['yellow_ratio'] = np.mean((r + g) / 2) # Approximation of yellow

    # 3. Texture (GLCM)
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    # Bounding box crop for efficiency
    gray_crop = gray[y:y+h, x:x+w]
    mask_crop = mask[y:y+h, x:x+w]
    
    # We set background pixels to 0 and exclude 0 from GLCM computation where possible,
    # or just compute it on the bounding box (standard practice).
    # Computing on the whole masked bounding box.
    glcm = graycomatrix(gray_crop, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                        levels=256, symmetric=True, normed=True)
    features['glcm_contrast'] = np.mean(graycoprops(glcm, 'contrast'))
    features['glcm_homogeneity'] = np.mean(graycoprops(glcm, 'homogeneity'))
    features['glcm_energy'] = np.mean(graycoprops(glcm, 'energy'))
    features['glcm_correlation'] = np.mean(graycoprops(glcm, 'correlation'))
    
    # 4. Spatial / Regional Features (3x3 Grid over bounding box)
    # We define 'Core' as the central 1/3x1/3 region, and 'Edge' as the rest.
    cell_w, cell_h = w // 3, h // 3
    core_x1, core_x2 = cell_w, 2 * cell_w
    core_y1, core_y2 = cell_h, 2 * cell_h
    
    core_mask = np.zeros_like(mask_crop)
    core_mask[core_y1:core_y2, core_x1:core_x2] = 255
    core_mask = cv2.bitwise_and(core_mask, mask_crop)
    
    edge_mask = cv2.subtract(mask_crop, core_mask)
    
    hsv_crop = img_hsv[y:y+h, x:x+w]
    rgb_crop = img_rgb[y:y+h, x:x+w]
    
    def get_region_stats(region_mask, prefix):
        if np.sum(region_mask) == 0:
            return {f'{prefix}_mean_H': 0, f'{prefix}_yellow_ratio': 0}
        reg_hsv = hsv_crop[region_mask > 0]
        reg_rgb = rgb_crop[region_mask > 0]
        mean_H = np.mean(reg_hsv[:, 0])
        
        R, G, B = reg_rgb[:,0].astype(float), reg_rgb[:,1].astype(float), reg_rgb[:,2].astype(float)
        denom = R + G + B
        denom[denom == 0] = 1
        y_ratio = np.mean((R/denom + G/denom)/2)
        return {f'{prefix}_mean_H': mean_H, f'{prefix}_yellow_ratio': y_ratio}

    features.update(get_region_stats(core_mask, 'core'))
    features.update(get_region_stats(edge_mask, 'edge'))
    
    return features


In [ ]:
def process_dataset(dataset_path):
    all_files = glob.glob(os.path.join(dataset_path, '*', '*.[jJ][pP][gG]'))
    print(f"Found {len(all_files)} images.")
    
    results = []
    
    for file_path in tqdm(all_files, desc="Processing Images"):
        basename = os.path.basename(file_path)
        class_name = os.path.basename(os.path.dirname(file_path))
        crop, condition = extract_crop_condition(class_name)
        
        img_bgr = cv2.imread(file_path)
        if img_bgr is None:
            continue
            
        try:
            segmented_rgb, mask, mask_ratio, quality_flag = segment_leaf(img_bgr)
            
            row_data = {
                'image_path': file_path,
                'image_name': basename,
                'class': class_name,
                'crop': crop,
                'condition': condition,
                'quality_flag': quality_flag,
                'mask_area_ratio': mask_ratio
            }
            
            if quality_flag == 'ACCEPT':
                feats = extract_all_features(segmented_rgb, mask)
                if feats is not None:
                    row_data.update(feats)
                else:
                    row_data['quality_flag'] = 'FAILED'
                    
            results.append(row_data)
        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue
        
    df = pd.DataFrame(results)
    return df

df_full = process_dataset(DATASET_PATH)
df_full.to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved {len(df_full)} records to {OUTPUT_CSV}")


Found 798 images.


Processing Images:   0%|          | 0/798 [00:00<?, ?it/s]

In [ ]:
df_accepted = df_full[df_full['quality_flag'] == 'ACCEPT']
df_failed = df_full[df_full['quality_flag'] != 'ACCEPT']

print('--- PIPELINE SUMMARY ---')
print(f'Total Processed: {len(df_full)}')
print(f'Accepted:        {len(df_accepted)}')
print(f'Failed/Excluded: {len(df_failed)}')
print()
print(f'Extracted {len(df_accepted.columns) - 7} numerical features per accepted image.')
print(f'Columns: {list(df_accepted.columns)}')


--- PIPELINE SUMMARY ---
Total Processed: 4749
Accepted:        4387
Failed/Excluded: 362

Extracted 38 numerical features per accepted image.
Columns: ['image_path', 'image_name', 'class', 'crop', 'condition', 'quality_flag', 'mask_area_ratio', 'area', 'perimeter', 'aspect_ratio', 'circularity', 'extent', 'solidity', 'convex_area', 'major_axis_length', 'minor_axis_length', 'eccentricity', 'mean_R', 'std_R', 'mean_G', 'std_G', 'mean_B', 'std_B', 'mean_H', 'std_H', 'mean_S', 'std_S', 'mean_V', 'std_V', 'mean_L', 'std_L', 'mean_a', 'std_a', 'mean_b', 'std_b', 'green_ratio', 'yellow_ratio', 'glcm_contrast', 'glcm_homogeneity', 'glcm_energy', 'glcm_correlation', 'core_mean_H', 'core_yellow_ratio', 'edge_mean_H', 'edge_yellow_ratio']
